# 01 - Exploration de Sen1Floods11

Objectif : comprendre à quoi ressemblent les données avant de construire le modèle.

- images Sentinel-2 (optique, 13 bandes)
- masques d'eau annotés à la main
- masques d'eau permanente (rivières, lacs) issus du JRC

Exécution : Google Colab, *Exécution > Tout exécuter*.

In [ ]:
!pip install -q rasterio

## Contenu du bucket public

In [ ]:
!gsutil ls gs://sen1floods11/v1.1/data/flood_events/HandLabeled/
!gsutil ls gs://sen1floods11/v1.1/splits/flood_handlabeled/

## Téléchargement (quelques minutes)

In [ ]:
!mkdir -p data
!gsutil -m -q cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand data/
!gsutil -m -q cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand data/
!gsutil -m -q cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/JRCWaterHand data/
!mkdir -p data/splits
!gsutil -m -q cp gs://sen1floods11/v1.1/splits/flood_handlabeled/*.csv data/splits/
!ls data/S2Hand | head -5 && ls data/S2Hand | wc -l
!ls data/splits

## Fonctions de lecture et d'affichage

In [ ]:
import rasterio, numpy as np, matplotlib.pyplot as plt, glob, os

def lire(chemin):
    with rasterio.open(chemin) as f:
        return f.read().astype("float32")   # forme (bandes, H, W)

def rgb(s2):
    # Sentinel-2 : bandes B1..B12 (+B8A). Rouge = B4 (idx 3), vert = B3 (idx 2), bleu = B2 (idx 1)
    img = s2[[3, 2, 1]].transpose(1, 2, 0)
    bas, haut = np.percentile(img, [2, 98])
    return np.clip((img - bas) / (haut - bas + 1e-6), 0, 1)

def ndwi(s2):
    # Indice d'eau de McFeeters : (vert - proche infrarouge) / (vert + PIR) = (B3 - B8) / (B3 + B8)
    vert, pir = s2[2], s2[7]
    return (vert - pir) / (vert + pir + 1e-6)

fichiers = sorted(glob.glob("data/S2Hand/*.tif"))
print(len(fichiers), "images")
with rasterio.open(fichiers[0]) as f:
    print("bandes :", f.count, "| taille :", f.width, "x", f.height, "| type :", f.dtypes[0])

## Quatre exemples : image, NDWI, eau annotée, eau permanente

In [ ]:
for chemin in fichiers[:4]:
    nom = os.path.basename(chemin).replace("_S2Hand.tif", "")
    s2 = lire(chemin)
    label = lire(f"data/LabelHand/{nom}_LabelHand.tif")[0]      # -1 = pas de donnée, 0 = sec, 1 = eau
    perm = lire(f"data/JRCWaterHand/{nom}_JRCWaterHand.tif")[0] # eau permanente

    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(rgb(s2));                                ax[0].set_title(nom)
    ax[1].imshow(ndwi(s2), cmap="RdBu", vmin=-1, vmax=1); ax[1].set_title("NDWI")
    ax[2].imshow(label, cmap="Blues", vmin=-1, vmax=1);   ax[2].set_title("Eau annotée")
    ax[3].imshow(perm, cmap="Greens");                    ax[3].set_title("Eau permanente")
    for a in ax: a.axis("off")
    plt.show()

## Statistiques par pays (pour le découpage sans fuite)

In [ ]:
import pandas as pd

lignes = []
for chemin in fichiers:
    nom = os.path.basename(chemin).replace("_S2Hand.tif", "")
    label = lire(f"data/LabelHand/{nom}_LabelHand.tif")[0]
    perm = lire(f"data/JRCWaterHand/{nom}_JRCWaterHand.tif")[0]
    valide = label >= 0
    eau = (label == 1) & valide
    inonde = eau & (perm != 1)           # eau qui n'est pas de l'eau permanente
    lignes.append({"image": nom,
                   "pays": nom.split("_")[0],
                   "part_eau": eau.sum() / max(valide.sum(), 1),
                   "part_inondee": inonde.sum() / max(valide.sum(), 1)})

stats = pd.DataFrame(lignes)
print(stats.groupby("pays")[["part_eau", "part_inondee"]].agg(["count", "mean"]).round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
stats["part_eau"].hist(bins=30, ax=ax[0]);     ax[0].set_xlabel("Part d'eau dans l'image")
stats["part_inondee"].hist(bins=30, ax=ax[1]); ax[1].set_xlabel("Part d'eau hors eau permanente")
plt.show()

## À retenir

- **NDWI** : l'eau apparaît en bleu ; ce sera la base de la baseline simple.
- **Eau permanente** : une rivière n'est pas une inondation. L'alerte portera sur l'eau *hors* eau permanente.
- **Pays** : les images sont groupées par événement ; le découpage se fera par pays pour tester sur des zones jamais vues.